## FP32 Baselines & Input Bit-Depth Sweep

For each trained checkpoint (`seed_1`, `seed_2`, `seed_42`), sweep FP32 and FP16 across all input bit-depths (8, 4, 2, 1).

Results are saved under `runs/val_infer/<seed>/`.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
PYFILES = PROJECT_ROOT / "pyfiles"
if str(PYFILES) not in sys.path:
    sys.path.insert(0, str(PYFILES))

In [2]:
import torch
import pandas as pd

from src.config import ExperimentConfig, with_overrides
from src.runner import run_experiment
from utils.utils import load_runs, flatten_runs

pd.set_option("display.max_columns", 200)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

torch: 2.10.0+cu130 | cuda: True


In [3]:
CKPT_ROOT = PROJECT_ROOT / "checkpoints"
INPUT_BITS = [8, 4, 2, 1]

checkpoints = {}
for bits in INPUT_BITS:
    ckpt_dir = CKPT_ROOT / f"fp32_{bits}bit"
    if not ckpt_dir.exists():
        continue
    for seed_dir in sorted(ckpt_dir.iterdir()):
        if seed_dir.is_dir() and (seed_dir / "best.pth").exists():
            checkpoints[(bits, seed_dir.name)] = str(seed_dir / "best.pth")

print("Checkpoints found:")
for (bits, seed), path in checkpoints.items():
    print(f"  {bits}bit / {seed}: {path}")


Checkpoints found:
  8bit / seed_1: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_8bit/seed_1/best.pth
  8bit / seed_2: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_8bit/seed_2/best.pth
  8bit / seed_42: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_8bit/seed_42/best.pth
  4bit / seed_1: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_4bit/seed_1/best.pth
  4bit / seed_2: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_4bit/seed_2/best.pth
  4bit / seed_42: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_4bit/seed_42/best.pth
  2bit / seed_1: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_2bit/seed_1/best.pth
  2bit / seed_2: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_2bit/seed_2/best.pth
  2bit / seed_42: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_2bit/seed_42/best.pth
  1bit / seed_1: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_1bit/seed_1/best

## FP32 Full Sweep (b=8, 4, 2, 1)

In [ ]:
fp32_records = []

for (bits, seed), ckpt_path in checkpoints.items():
    base = ExperimentConfig(
        backend="pytorch",
        device="cuda",
        batch_size=1,
        seed=42,
        num_eval_batches=500,
        output_root=f"../runs/val/ptq/{seed}",
    )
    cfg = with_overrides(base, model_precision="fp32", input_quant_bits=bits)

    result_path = cfg.result_json_path()
    if result_path.exists():
        import json
        with open(result_path) as f:
            payload = json.load(f)
        print(f"  SKIP fp32 {bits}bit / {seed} (loaded from disk)")
    else:
        print(f"\n{'='*60}")
        print(f"  {bits}bit / {seed}  |  precision: fp32")
        print(f"{'='*60}")
        payload, _ = run_experiment(cfg, save_results_flag=True, checkpoint_path=ckpt_path)

    fp32_records.append(payload)
    print(f"  top1={payload['results']['top1_acc']:.2f}%  top5={payload['results']['top5_acc']:.2f}%")

## FP16 Full Sweep (b=8, 4, 2, 1)

In [ ]:
fp16_records = []

for (bits, seed), ckpt_path in checkpoints.items():
    base = ExperimentConfig(
        backend="pytorch",
        device="cuda",
        batch_size=1,
        seed=42,
        num_eval_batches=500,
        output_root=f"../runs/val/ptq/{seed}",
    )
    cfg = with_overrides(base, model_precision="fp16", input_quant_bits=bits)

    result_path = cfg.result_json_path()
    if result_path.exists():
        import json
        with open(result_path) as f:
            payload = json.load(f)
        print(f"  SKIP fp16 {bits}bit / {seed} (loaded from disk)")
    else:
        print(f"\n{'='*60}")
        print(f"  {bits}bit / {seed}  |  precision: fp16")
        print(f"{'='*60}")
        payload, _ = run_experiment(cfg, save_results_flag=True, checkpoint_path=ckpt_path)

    fp16_records.append(payload)
    print(f"  top1={payload['results']['top1_acc']:.2f}%  top5={payload['results']['top5_acc']:.2f}%")

## Results Summary

In [ ]:
all_rows = []
for bits in INPUT_BITS:
    ckpt_dir = CKPT_ROOT / f"fp32_{bits}bit"
    if not ckpt_dir.exists():
        continue
    for seed_dir in sorted(ckpt_dir.iterdir()):
        if not seed_dir.is_dir():
            continue
        run_dir = f"../runs/val/ptq/{seed_dir.name}"
        runs = load_runs(run_dir, status="ok")
        for r in flatten_runs(runs):
            r["seed"] = seed_dir.name
            r["input_bits_trained"] = bits
            all_rows.append(r)

df = pd.DataFrame(all_rows)

display_cols = [
    "seed",
    "input_bits_trained",
    "run_id",
    "cfg.model_precision",
    "cfg.input_quant_bits",
    "res.top1_acc",
    "res.top5_acc",
    "res.infer_ms_avg",
    "res.throughput_infer_sps",
    "res.total_samples",
]

df[display_cols].sort_values(
    ["input_bits_trained", "seed", "cfg.model_precision", "cfg.input_quant_bits"],
    ascending=[False, True, True, False],
)

In [12]:
avg_df = df[df["cfg.backend"] == "pytorch"].groupby(
    ["cfg.backend", "cfg.model_precision", "input_bits_trained"]
).agg(
    top1_mean=("res.top1_acc", "mean"),
    top1_std=("res.top1_acc", "std"),
    top5_mean=("res.top5_acc", "mean"),
    top5_std=("res.top5_acc", "std"),
    infer_ms_mean=("res.infer_ms_avg", "mean"),
    infer_ms_std=("res.infer_ms_avg", "std"),
    throughput_mean=("res.throughput_infer_sps", "mean"),
).round(3).reset_index()

avg_df = avg_df.sort_values(
    ["cfg.model_precision","input_bits_trained"],
    ascending=[True, True],
).reset_index(drop=True)
avg_df


,cfg.backend,cfg.model_precision,input_bits_trained,top1_mean,top1_std,top5_mean,top5_std,infer_ms_mean,infer_ms_std,throughput_mean
0,pytorch,fp16,1,86.028,5.809,97.872,2.221,3.108,0.151,322.282
1,pytorch,fp16,2,90.355,5.108,99.220,1.351,3.018,0.169,332.010
2,pytorch,fp16,4,94.681,6.279,99.645,0.614,2.972,0.002,336.435
3,pytorch,fp16,8,94.468,7.015,99.574,0.737,3.012,0.249,333.605
4,pytorch,fp32,1,86.028,5.809,97.801,2.152,2.896,0.003,345.304
5,pytorch,fp32,2,90.426,5.177,99.220,1.351,3.018,0.170,332.040
6,pytorch,fp32,4,94.610,6.402,99.645,0.614,2.958,0.157,338.740
7,pytorch,fp32,8,94.468,7.015,99.574,0.737,2.994,0.064,334.138


In [ ]:
import json, numpy as np

bench_path = PROJECT_ROOT / "resultsv2" / "latency_bench" / "pytorch_fp32_in8b_cuda_bs1.json"
with open(bench_path) as f:
    bench = json.load(f)
bench_std = np.std(bench["latencies_ms"], ddof=1)

avg_df[["cfg.model_precision", "input_bits_trained",
        "top1_mean", "top1_std", "top5_mean", "top5_std",
        "infer_ms_mean"]].assign(
    top1=lambda d: d.apply(lambda r: f"{r['top1_mean']:.2f} ± {r['top1_std']:.2f}", axis=1),
    top5=lambda d: d.apply(lambda r: f"{r['top5_mean']:.2f} ± {r['top5_std']:.2f}", axis=1),
    infer_ms=lambda d: d.apply(lambda r: f"{r['infer_ms_mean']:.3f} ± {bench_std:.3f}", axis=1),
)[["cfg.model_precision", "input_bits_trained", "top1", "top5", "infer_ms"]]

In [ ]:
out_path = PROJECT_ROOT / "resultsv2" / "val_runs" / "ptq_torch_avg_val.json"
avg_df.to_json(out_path, orient="records", indent=2)
print(f"Saved to {out_path}")